<div style="display: flex; background-color: RGB(89,0,22);" >
<h1 style="margin: auto; padding: 30px; ">ANALYSE DU STOCK ET DES VENTES DU SITE BOTTLENECK</h1>
</div>

# OBJECTIF DE CE NOTEBOOK

Bienvenue dans l'outil plébiscité par les analystes de données Jupyter.

Il s'agit d'un outil permettant de mixer et d'alterner code, texte et graphiques.

Cet outil est formidable pour plusieurs raisons:

+ Il permet de tester des lignes de codes au fur et à mesure de votre rédaction, de constater immédiatement le résultat d'une instruction, de la corriger si nécessaire.
+ Il permet aussi de rédiger du texte pour expliquer l'approche suivie ou les résultats d'une analyse et de le mettre en forme grâce à du code html ou plus simple avec **Markdown**
+ Il est possible d'ajouter des graphiques

Pour vous aider dans vos premiers pas à l'usage de Jupyter et de Python, nous avons rédigé ce notebook en vous indiquant les instructions à suivre.

Il vous suffit pour cela de saisir le code Python répondant à l'instruction donnée.

Vous verrez de temps à autre le code Python répondant à une instruction donnée mais cela est fait pour vous aider à comprendre la nature du travail qui vous est demandé.

Et gardez à l'esprit qu'il n'y a pas de solution unique pour résoudre un problème et qu'il y a autant de résolutions de problèmes que de développeurs ;)...



<div style="background-color: RGB(0, 128, 96);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 1 - Importation des librairies et chargement des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">1.1 - Importation des librairies</h3>
</div>

!pip install "kaleido==0.2.1"

#Importation de la librairie Pandas
import pandas as pd

#Importation librairies complémentaires
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap, Normalize
import matplotlib.ticker
from matplotlib.ticker import FuncFormatter
import seaborn as sb

#Importation de la librairie plotly express
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"


#Trouver dans Google l'instruction permettant d'afficher toutes les colonnes d'un dataframe
#Saisir dans Google les mots clés "display all columns dataframe Pandas" par exemple.
#Dans les résultats de la recherche, privilégier les solutions provenant de Stack Overflow ou Medium

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
   

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import pandera.pandas as pa
from sklearn.ensemble import IsolationForest

pio.renderers.default = "png"      # affiche les graphiques plotly en image statique
pd.set_option("display.max_columns", None)  # affiche toutes les colonnes d'un tableau, sans troncature

SEED = 42                          # fige une valeur de référence pour rendre l'aléatoire reproductible
DATA = Path("data")                # chemin relatif vers le dossier des fichiers sources

<div style="border: 3px solid #970e97a4" >
<h5>Optimisation:</h5>

</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">1.2 - Chargements des fichiers</h3>
</div>

#Importation du fichier web.xlsx
df_web = pd.read_excel("C:/Users/chris/Documents/GitHub/OC/P6/data_raw/web.xlsx")
#Importation du fichier erp.xlsx
df_erp = pd.read_excel("C:/Users/chris/Documents/GitHub/OC/P6/data_raw/erp.xlsx")
#Importation du fichier liaison.xlsx
df_liaison = pd.read_excel("C:/Users/chris/Documents/GitHub/OC/P6/data_raw/liaison.xlsx")

In [2]:
df_erp = pd.read_excel(DATA / "erp.xlsx")
df_web = pd.read_excel(DATA / "web.xlsx")
df_liaison = pd.read_excel(DATA / "liaison.xlsx")

for nom, df in [("erp", df_erp), ("web", df_web), ("liaison", df_liaison)]:
    print(f"{nom:8} {df.shape[0]:>5} lignes  {df.shape[1]} colonnes")
    print(f"         colonnes : {list(df.columns)}\n")

c:\Users\chris\Documents\GitHub\OC\P13\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


erp        825 lignes  6 colonnes
         colonnes : ['product_id', 'onsale_web', 'price', 'stock_quantity', 'stock_status', 'purchase_price']

web       1513 lignes  29 colonnes
         colonnes : ['sku', 'virtual', 'downloadable', 'rating_count', 'average_rating', 'total_sales', 'tax_status', 'tax_class', 'post_author', 'post_date', 'post_date_gmt', 'post_content', 'product_type', 'post_title', 'post_excerpt', 'post_status', 'comment_status', 'ping_status', 'post_password', 'post_name', 'post_modified', 'post_modified_gmt', 'post_content_filtered', 'post_parent', 'guid', 'menu_order', 'post_type', 'post_mime_type', 'comment_count']

liaison    825 lignes  2 colonnes
         colonnes : ['id_web', 'product_id']



c:\Users\chris\Documents\GitHub\OC\P13\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\chris\Documents\GitHub\OC\P13\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


<div style="border: 3px solid #970e97a4" >
<h5>Optimisation:</h5>

</div>

<div style="background-color: RGB(0, 128, 96);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 2 - Analyse exploratoire des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1 - Analyse exploratoire du fichier erp.xlsx</h3>
</div>

#Afficher les dimensions du dataset
print("Le tableau comporte {} observations".format(df_erp.shape[0]))
print("Le tableau comporte {} colonnes".format(df_erp.shape[1]))

#Consulter le nombre de colonnes
print(df_erp.columns)
#La nature des données dans chacune des colonnes
print("La nature des colonnes est de type: {}".format(df_erp.dtypes))
#Le nombre de valeurs présentes dans chacune des colonnes
print("Voici le nombre de valeurs présentent dans chaque colonnes:{}".format(df_erp.count()))

#Afficher les 5 premières lignes de la table
df_erp.head(5)


#Vérifier si il y a des lignes en doublon dans la colonne product_id
dup_erp=df_erp[df_erp.duplicated('product_id', keep=False)]
print(dup_erp)

#Afficher les valeurs distinctes de la colonne stock_status
df_erp['stock_status'].unique()

*À quelle(s) autre(s) colonne(s) sont-elles liées ?*


La colonne stock-status est liée aux colonnes onsale_web et stock-quantity.

Onsale_web décompte le nombre de vente de stock-quantity. Lorsque cette derniere est à zéro le stock-status change pour "outofstock".

#Création d'une colonne "stock_status_2"
#La valeur de cette deuxième colonne sera fonction de la valeur dans la colonne "stock_quantity"
#Si la valeur de la colonne "stock_quantity" est nulle, renseigner "outofstock" sinon mettre "instock"
df_erp['stock_status_2']=np.where(df_erp['stock_quantity']==0, 'outofstock', 'instock')


#Vérifions que les 2 colonnes sont identiques:
#Les 2 colonnes sont strictement identiques si les valeurs de chaque ligne sont strictement identiques 2 à 2
#La comparaison de 2 colonnes peut se réaliser simplement avec l'instruction ci-dessous:
df_erp["stock_status"] == df_erp["stock_status_2"]

#Le résultat est l'affichage de True ou False pour chacune des lignes du dataset
#C'est un bon début, mais difficile à exploiter

#Mais il est possible de synthétiser ce résultat en effectuant la somme de cette colonne:
#True vaut 1 et False 0
#Nous devrions obtenir la somme de 824 qui correspond au nombre de lignes dans ce dataset
dup=(df_erp["stock_status"] == df_erp["stock_status_2"]).sum()
print(dup)


#Si les colonnes ne sont absolument pas identiques ligne à ligne alors identifier la ligne en écart
##Dans ce cas je vous donne ce lien pour apprendre à réaliser des filtres dans Pandas:
##https://bitbucket.org/hrojas/learn-pandas/src/master/
##Lesson 3
df_erp['dup']=df_erp["stock_status"] == df_erp["stock_status_2"]
sortdf_erp = df_erp[df_erp['dup']==False].sort_index(axis=0)
sortdf_erp.head(10)

#Corriger la ou les données incohérentes
df_erp['stock_status']=np.where(~df_erp['dup'],df_erp['stock_status_2'],df_erp['stock_status'])
#Vérification en utilisant le même code que plus haut pour afficher les problèmes
dup=(df_erp["stock_status"] == df_erp["stock_status_2"]).sum()
print(dup)

sortdf_erp = df_erp[df_erp['dup']==False].sort_index(axis=0)
sortdf_erp.head(10)


In [3]:
print(f"erp : {df_erp.shape[0]} lignes, {df_erp.shape[1]} colonnes")
print(df_erp.dtypes, "\n")
print("Doublons product_id:", df_erp.duplicated('product_id').sum())

# stock_status vs stock_quantity : vérification de cohérence avant suppression de la colonne
stock_status_recalcule = np.where(df_erp['stock_quantity'] == 0, 'outofstock', 'instock')
incoherent_status = df_erp['stock_status'] != stock_status_recalcule
print("Lignes stock_status incohérentes:", incoherent_status.sum())

# Repérage des erreurs, avant correction
mask_price = df_erp['price'] < 0
mask_stock = df_erp['stock_quantity'] < 0

erreurs = df_erp.loc[incoherent_status | mask_price | mask_stock,
                      ['product_id', 'stock_status', 'stock_quantity', 'price']].copy()
erreurs['stock_status_incoherent'] = incoherent_status
erreurs['price_negatif'] = mask_price
erreurs['stock_negatif'] = mask_stock
print(f"\n{len(erreurs)} product_id avec au moins une anomalie :")
print(erreurs)

# Corrections — volumes confirmés ci-dessus et par le schéma Pandera (axe 1)
print(f"\nprice négatifs corrigés : {mask_price.sum()}")
df_erp['price'] = df_erp['price'].abs()

print(f"stock_quantity négatifs corrigés : {mask_stock.sum()}")
df_erp.loc[mask_stock, 'stock_quantity'] = 0

# onsale_web : hors périmètre (statut promo) ; stock_status : redondant avec stock_quantity, vérifié ci-dessus
df_erp = df_erp.drop(columns=['onsale_web', 'stock_status'])

print(f"\npurchase_price : min {df_erp['purchase_price'].min():.2f}, max {df_erp['purchase_price'].max():.2f}")
df_erp.head()

erp : 825 lignes, 6 colonnes
product_id          int64
onsale_web          int64
price             float64
stock_quantity      int64
stock_status          str
purchase_price    float64
dtype: object 

Doublons product_id: 0
Lignes stock_status incohérentes: 4

7 product_id avec au moins une anomalie :
     product_id stock_status  stock_quantity  price  stock_status_incoherent  \
4          4039   outofstock               3   46.0                     True   
151        4233   outofstock               0  -20.0                    False   
398        4885      instock               0   18.7                     True   
449        4973   outofstock             -10   10.0                     True   
469        5017   outofstock               0   -8.0                    False   
573        5700   outofstock              -1   44.5                     True   
739        6594      instock              19   -9.1                    False   

     price_negatif  stock_negatif  
4            False  

,product_id,price,stock_quantity,purchase_price
0,3847,24.2,16,12.88
1,3849,34.3,10,17.54
2,3850,20.8,0,10.64
3,4032,14.1,26,6.92
4,4039,46.0,3,23.77


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1 - Analyse exploratoire de chaque variable du fichier erp.xlsx</h3>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1.1 - Analyse de la variable PRIX</h3>
</div>

In [ ]:
###############
## LES PRIX  ##
###############

#Vérification des prix: Y a t-il des prix non renseignés, négatifs ou nuls?
#Afficher le ou les prix non renseignés dans la colonne "price"
print("Nombres d'articles avec un prix non renseigné: {}".format((df_erp['price']=='').sum())) #Saisir l'instruction manquante dans la fonction format
#Afficher le prix minimum de la colonne "price"
print("Prix min: {}".format(df_erp['price'].min())) #Saisir l'instruction manquante dans la fonction format
#Afficher le prix maximum de la colonne "price"
print("Prix max: {}".format(df_erp['price'].max())) #Saisir l'instruction manquante dans la fonction format
#Afficher les prix inférieurs à 0 (qu'est-ce qu'il faut en faire ?)
neg_price=df_erp[df_erp['price']<0]
neg_price.head(10)

*La marge appliquée aux autres produits semblent correspondre a celle des prix erronés.
Le choix a été fait de repasser les prix en positifs afin de les intégrer à l'analyse.*

In [ ]:
#Transformer les chiffres négatifs en positifs
df_erp['price']=df_erp['price'].abs()
neg_price=df_erp[df_erp['price']<0]

print("Prix min: {}".format(df_erp['price'].min())) 

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1.2 - Analyse de la variable STOCK</h3>
</div>

In [ ]:
#######################
### stock_quantity  ###
#######################

#Vérification de la colonne stock quantity
#Afficher la quantité minimum de la colonne "stock_quantity"
print("Stock min: {}".format(df_erp['stock_quantity'].min()))
#Afficher la quantité maximum de la colonne "stock_quantity"
print("Stock max: {}".format(df_erp['stock_quantity'].max()))
#Afficher les stocks inférieurs à 0 (qu'est-ce qu'il faut en faire ?)
neg_stock=df_erp[df_erp['stock_quantity']<0]
neg_stock.head()

*Il s'agit soit d'une erreur de saisie, soit d'un report de stock qui accepte les stocks négatif. Ici le choix a été fait de repasser le stcok à zéro.*

In [ ]:
#Transformer les chiffres négatifs en o
mask=df_erp['stock_quantity']<0
df_erp.loc[mask, 'stock_quantity']=0
#Afficher la quantité minimum de la colonne "stock_quantity"
print("Stock min: {}".format(df_erp['stock_quantity'].min()))

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1.3 - Analyse de la variable ONSALE_WEB</h3>
</div>

*Vérification de la colonne onsale_web et des valeurs qu'elle contient. Que signifient-elles?*

onsale_web ne contient que  deux valeurs 0 et 1.
Cette colonne désigne un statut promotionnel, ici c'est un boléen avec 1 signifiant que la promo est active coté web avec un affichage particulier.

*Quelles sont les colonnes à conserver selon vous?*

Toutes à l'exception de onsale_web, dup, stock_status et stock_status_2

In [ ]:
#Supprimer la colonne comportant le libellé "stock_status_2" car elle est redondante 
#avec la colonne "stock_status".
df_erp.drop(columns=['onsale_web','stock_status','stock_status_2', 'dup'], inplace=True)
df_erp.head(5)

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1.4 - Analyse de la variable prix d'achat</h3>
</div>

In [ ]:
######################
##   prix d'achat   ##
######################

#Vérification de la colonne purchase_price : 
#Afficher le ou les prix non renseignés dans la colonne "purchase_price"
print("Nombres d'articles avec un prix non renseigné: {}".format((df_erp['purchase_price']=='').sum()))
#Afficher le prix minimum de la colonne "purchase_price"
print("Prix d'achat min: {}".format(df_erp['purchase_price'].min()))
#Afficher le prix maximum de la colonne "purchase_price"
print("Prix d'achat max: {}".format(df_erp['purchase_price'].max()))

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.2 - Analyse exploratoire du fichier web.xlsx</h3>
</div>
 

In [ ]:
#Dimension du dataset
#Nombre d'observations
print("Le tableau comporte {} observations".format(df_web.shape[0]))
#Nombre de caractéristiques
print("Le tableau comporte {} colonnes".format(df_web.shape[1]))


In [ ]:
#Consulter le nombre de colonnes
#La nature des données dans chacune des colonnes
#Le nombre de valeurs présentes dans chacune des colonnes
print(df_web.columns)
print("La nature des colonnes est de type: {}".format(df_web.dtypes))
print("Voici le nombre de valeurs présentent dans chaque colonnes:{}".format(df_web.count()))

*Selon vous, quelles sont les colonnes à conserver ?*


sku et total_sales et product_type

In [ ]:
#Si vous avez défini des colonnes à supprimer, effectuer l'opération
df_web.drop(columns=['virtual','downloadable','rating_count','average_rating','tax_status','tax_class','post_author','post_date','post_date_gmt','post_content','post_title','post_excerpt','post_status','comment_status','ping_status','post_password','post_name','post_modified','post_modified_gmt','post_content_filtered','post_parent','guid','menu_order','post_type','post_mime_type','comment_count'], inplace=True)
df_web.head(5)


In [ ]:
#Visualisation des valeurs de la colonne sku
df_web['sku'].unique()

*Quelles sont les valeurs qui ne semblent pas respecter la régle de codification?*

Les valeurs sans respect de la règle de codification sont celles qui ne sont pas composées uniquement de chiffres.

In [ ]:
#Identifier les lignes sans code article
s=df_web['sku'].astype('string').str.strip()#stockage de la colonne sku en tant que chaîne de caractères et suppression des espaces éventuels
sku_invalid = ~s.str.fullmatch(r'\d+') #~ signifie "non", str.fullmatch(r'\d+') vérifie que la chaîne est composée uniquement de chiffres
df_web_invalid = df_web.loc[sku_invalid, ['sku']]
df_web_invalid.head()

In [ ]:
#Pour les codes articles identifiés, réaliser une analyse et définir l'action à entreprendre
#Suppression des lignes non correspondantes au code défini car impossible à retracer sur le fichier erp.
df_web.drop(df_web_invalid.index, inplace=True)
df_web.reset_index(drop=True, inplace=True)

In [ ]:
#La clé pour chaque ligne est-elle unique? autrement dit, y a-t-il des doublons?
dup_web=df_web[df_web.duplicated('sku', keep=False)] #Les valeurs nans sont considérées comme des doublons
n_dup = dup_web.shape[0]
print("Nombre de lignes en doublon selon la clé sku: {}".format(n_dup))

In [ ]:
#Suppression des doublons en gardant la première occurrence
df_web.drop_duplicates(subset='sku', keep='first', inplace=True)
#Comptage du nombre de ligne pour 'sku'
n_sku = df_web['sku'].shape[0]
print("Nombre de lignes après suppression des doublons selon la clé sku: {}".format(n_sku)) 

In [ ]:
#Les lignes sans code article semblent être toutes non renseignées
#Pour s'en assurer, réaliser les étapes suivantes:
#1 - Créer un dataframe avec uniquement les lignes sans code article
df_webna=df_web[df_web['sku'].isna()]
#2 - Utiliser la fonction df.info() sur ce nouveau dataframe pour observer le nombre de valeurs renseignées dans chacune des colonnes
df_webna.info()

*3 - Que constatez-vous?*


Sur les 85 lignes retournées seules 2 lignes ont des valeurs renseignées dans d'autres colonnes. Les autres lignes sont vides à l'exception de la colonne "sku".

In [ ]:
#Suppression des lignes vides dans le dataframe principal
df_web.dropna(how='all', inplace=True) #how='all' supprime les lignes où toutes les valeurs sont NaN
print("Voici le nombre de valeurs présentent dans chaque colonnes:{}".format(df_web.count()))

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.3 - Analyse exploratoire du fichier liaison.xlsx</h3>
</div>

In [ ]:
#Dimension du dataset
#Nombre d'observations
print("Le tableau comporte {} observations".format(df_liaison.shape[0]))
print("Le tableau comporte {} colonnes".format(df_liaison.shape[1]))
#Nombre de caractéristiques


In [ ]:
#Consulter le nombre de colonnes
print(df_liaison.columns)
#La nature des données dans chacune des colonnes
print("La nature des colonnes est de type: {}".format(df_liaison.dtypes))
#Le nombre de valeurs présentes dans chacune des colonnes
print("Voici le nombre de valeurs présentent dans chaque colonnes:{}".format(df_liaison.count()))



In [ ]:
#Les valeurs de la colonne "product_id" sont-elles toutes uniques?
df_liaison[df_liaison.duplicated('product_id', keep=False)]

In [ ]:
#Les valeurs de la colonne "id_web" sont-elles toutes uniques?
df_liaison[df_liaison.duplicated('id_web', keep=False)]

*Avons-nous des articles sans correspondance?*  

Les valeurs de product_id et id_web sont unique mais beaucoup de d'id_web sont des nan.
Il y a donc des articles sans correspondance.

In [ ]:
liaison_na=df_liaison['id_web'].isna().sum()
print("Nombre de lignes sans correspondance: {}".format(liaison_na)) 

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 3 - Jonction des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 3.1 - Jonction du fichier df_erp et df_liaison</h3>
</div>

In [ ]:
#Fusion des fichiers df_erp et df_liaison
df_merge=pd.merge(df_erp, df_liaison, on='product_id', how='inner')
df_merge.head(5)

In [ ]:
#Y a t-il des lignes ne "matchant" pas entre les 2 fichiers?
df_na=df_merge[df_merge['id_web'].isna()]
count_na=df_na['product_id'].count()
print("Nombre de lignes sans correspondance entre les 2 fichiers: {}".format(count_na))


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 3.2 - Jonction du fichier df_merge et df_web</h3>
</div>

In [ ]:
#Fusionner les datasets df_merge et df_web
df_stats=pd.merge(df_merge, df_web, left_on='id_web', right_on='sku', how='inner')
df_stats.head(5)

In [ ]:
#Avons-nous des lignes sans correspondance?
mask=(df_stats['id_web'].isna() | df_stats['product_id'].isna())
df_nan = df_stats.loc[mask].shape[0]
print("Nombre de lignes sans correspondance: {}".format(df_nan)) 


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 4 - Analyse univariée des prix</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.1 - Exploration par la visualisation de données</h3>
</div>

In [ ]:
#Création d'une boîte à moustache de la répartition des prix grâce à Pandas
ax=df_stats.boxplot(column="price")

ax.set_title('Distribution des prix des produits')
ax.set_ylabel('Prix en euros')
ax.set_xlabel('')

plt.show()

In [ ]:
#Autre méthode avec plotly express
import plotly.express as px
fig = px.box(df_stats['price'].dropna(), y=df_stats['price'].dropna(), points='outliers', title='Distribution des prix')
fig.update_layout(yaxis_title='Prix en euros')
fig.show()

Globalement la majorité des produits ont un prix compris entre 5.2e et 83.7e.

Le maximum est de 225 euros et le minimum de 5.2 euros.

Une dizaine d'outliers ont un prix réparti au dela de 84 euros.

In [ ]:
#Pie chart

couleurs_produits={
    'Whisky':'saddlebrown',
    'Champagne':'darkgoldenrod',
    'Cognac':'palegoldenrod',
    'Vin':'darkred',
    'Gin':'black',
    'Huile d\'olive':'olive'
} 
fig=px.pie(df_stats,
            names='product_type',
           color='product_type', 
           color_discrete_map=couleurs_produits, 
           title='Répartition des types de produits du catalogue ')
fig.update_traces(textposition='outside', textinfo='percent+label')
fig.show()



<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.2 - Exploration par l'utilisation de méthodes statistiques</h3>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.2.1 - Identification par le Z-index</h3>
</div>

In [ ]:
#Calculer la moyenne du prix
avg_price=df_stats['price'].mean()
print("La moyenne des prix est de: {:.2f} ".format(avg_price))#{:.2f} correspond à 2 décimales
#Calculer l'écart-type du prix
et_price=df_stats['price'].std()
print("L'écart-type des prix est de: {:.2f} ".format(et_price))#L'écart-type permet de mesurer la dispersion des valeurs autour de la moyenne.Plus il est grand, plus les valeurs sont dispersées.
#Calculer le Z-score
zscore_price=(df_stats['price']-avg_price)/et_price 
df_stats['zscore_price']=zscore_price
df_stats.head(5)

Le z-score permet de mesurer l'écart d'une valeur par rapport à la moyenne en termes d'écart-type.Plus le z-score est élevé, plus la valeur est éloignée de la moyenne.

In [ ]:
#Quel est le seuil prix dont le z-score est supérieur à 3?
df_z3=df_stats[df_stats['zscore_price']>3].sort_values(by='zscore_price', ascending=True)
print(df_z3.iloc[0])


In [ ]:
df_z2=df_stats[df_stats['zscore_price']>2].sort_values(by='zscore_price', ascending=True)
df_z2.reset_index(drop=True, inplace=False)
df_z2.head(100)


Il s'agit d'une bouteille de vin dont le prix est très élevé par rapport à la moyenne des prix des produits du catalogue.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.2.2 - Identification par l'intervalle interquartile</h3>
</div>

In [ ]:
print(df_stats['price'].describe())

In [ ]:
#Définir un seuil pour les articles "outliers" en prix
#IQR=Q3-Q1, Bornes inférieure et supérieure = Q1 - 1.5*IQR et Q3 + 1.5*IQR.

q1, q3=df_erp['price'].quantile([0.25, 0.75])
iqr=q3-q1
low, high=q1-1.5*iqr, q3+1.5*iqr
mask=(df_stats['price'] < low) | (df_stats['price'] > high)
df_mask=df_stats.loc[mask].sort_values(by='price', ascending=True)
df_mask.head(10)

In [ ]:
#Définir le nombre d'articles et la proportion de l'ensemble du catalogue "outliers"
print("Nombre d'articles outliers: {}".format(df_mask.shape[0]))
proportion_outliers=(df_mask.shape[0]/df_stats.shape[0])*100
print("Proportion d'articles outliers dans le catalogue: {:.2f} %".format(proportion_outliers))

*Selon vous, ces outliers sont-ils justifiés ? Comment le démontrer si cela est possible ?*

Un outlier qui n'est ni une erreur de prix, ni un cas artificiel est justifié.
Plusieurs raisons sont possibles : rareté du produit, prix d'achat élevé, promotion, saisonnalité, etc...

In [ ]:
#Pie chart des articles outliers vs non outliers

fig=px.pie(df_mask,
            names='product_type',
           color='product_type', 
           color_discrete_map=couleurs_produits, 
           title='Proportion des articles outliers par type de produit ')
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 5 - Analyse univariée du CA, des quantités vendues, des stocks et de la marge ainsi qu'une analyse multivariée  </h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.1 - Analyse des ventes en CA</h3>
</div>

In [ ]:
##############################
# Calculer le CA du site web #
##############################

#Créer une colonne calculant le CA par article
df_stats['ca_par_article']=df_stats['price']*df_stats['total_sales']
#Calculer la somme de la colonne "ca_par_article"
#Ce résultat correspond au chiffre d'affaire du site web
ca_total=df_stats['ca_par_article'].sum()
print("Le chiffre d'affaire du site web est de: {:.2f} ".format(ca_total))

In [ ]:
#Graphique du chiffre d'affaire par type de produit
df_ca_produit=df_stats.groupby('product_type', as_index=False)['ca_par_article'].sum()
fig = px.bar(df_ca_produit, x='product_type', y='ca_par_article',
            color='product_type',
            color_discrete_map=couleurs_produits,
             title='Chiffre d\'affaire par type de produit',
             labels={'product_type':'Type de produit', 'ca_par_article':"Chiffre d'affaire en euros"})
fig.update_layout(legend_title_text='Type de produit')
fig.show()

In [ ]:
df_no_out=df_stats.loc[~mask].copy()
ca_no_out=df_no_out['ca_par_article'].sum()
print("Le chiffre d'affaire du site web sans les outliers est de: {:.2f} ".format(ca_no_out))
diff = ca_total - ca_no_out
print("Les outliers représentent donc un CA de {:.2f} (soit {:.2f} %)".format(diff, diff/ca_total*100))


In [ ]:
###############################
# Palmarès des articles en CA #
###############################

#Effectuer le tri dans l'ordre décroissant du CA du dataset df_merge
df_stats=df_stats.sort_values(by='ca_par_article', ascending=False)
#Réinitialiser l'index du dataset par un reset_index
df_stats.reset_index(drop=True, inplace=True)
#Afficher les 20 premiers articles en CA
df_stats.head(20)

In [ ]:
s1 = df_stats.sort_values('ca_par_article', ascending=False)['product_id'].head(20).reset_index(drop=True)
s2 = df_no_out.sort_values('ca_par_article', ascending=False)['product_id'].head(20).reset_index(drop=True)
diff = s1.eq(s2).sum()
print("Nombre d'articles avec outliers: {}".format(diff))

In [ ]:
#Créer le diagramme en barre des 20 premiers articles avec plotly express
df_stats20 = df_stats.nlargest(20, 'ca_par_article').copy() #nlargest permet de trier et sélectionner les n plus grandes valeurs d'une colonne
df_stats20['xlabel']=df_stats20['product_id'].astype(str)
fig = px.bar(df_stats20, 
             x='xlabel', y='ca_par_article',
            color='product_type', 
            color_discrete_map=couleurs_produits,
            text='price',
             title='Classement des 20 articles générant le plus de CA ',
             labels={'xlabel':'Article', 'ca_par_article':"Chiffre d'affaire en euros"})
fig.update_xaxes(categoryorder='array', categoryarray=df_stats20['xlabel'])
fig.update_traces(texttemplate="%{text:.2f}€")
fig.update_layout(legend_title_text='Type de produit')
fig.show()


In [ ]:
df_stats20.describe()

In [ ]:
#############################
# Calculer le 20 / 80 en CA #
#############################

#Créer une colonne calculant la part du CA de la ligne dans le dataset
df_stats['part_ca']=df_stats['ca_par_article']/ca_total
df_stats.head(20)

In [ ]:
#Créer une colonne réalisant la somme cumulative de la colonne précedemment créée
df_stats['ca_cumulee']=df_stats['part_ca'].cumsum()
df_stats.head()


In [ ]:
#Grâce aux deux colonnes créées précedemment, calculer le nombre d'articles représentant 80% du CA
art80=df_stats[df_stats['ca_cumulee']<=0.8].shape[0]
print("Nombre d'articles représentant 80% du CA: {}".format(art80))
#Afficher la proportion que représente ce groupe d'articles dans le catalogue entier du site web
print("Proportion des articles représentant 80% du CA dans le catalogue: {:.2f} %".format((art80/df_stats.shape[0])*100))
artvin=df_stats[(df_stats['ca_cumulee']<=0.8) & (df_stats['product_type']=='Vin')].shape[0]
print("Nombre d'articles de vin représentant 80% du CA: {}".format(artvin))
artchamp=df_stats[(df_stats['ca_cumulee']<=0.8) & (df_stats['product_type']=='Champagne')].shape[0]
print("Nombre d'articles de champagne représentant 80% du CA: {}".format(artchamp))
artpcvin=(artvin/art80)*100
print("Proportion des articles de vin représentant 80% du CA: {:.2f} %".format(artpcvin))
artpchamp=(artchamp/art80)*100
print("Proportion des articles de champagne représentant 80% du CA: {:.2f} %".format(artpchamp))

#Nombre de produits par type de produit
prod_counts = df_stats['product_type'].value_counts()
print("Nombre de produits par type de produit:\n{}".format(prod_counts))

In [ ]:
df_pareto = df_stats.sort_values('ca_cumulee', ascending=True).copy()
df_pareto['product_id'] = df_pareto['product_id'].astype(str)  # réassignation !

import plotly.graph_objects as go

fig = go.Figure()

# Barres CA
fig.add_trace(go.Bar(
    x=df_pareto['product_id'],
    y=df_pareto['ca_par_article'],
    name="Chiffre d'affaires"
))

# Courbe part cumulée
fig.add_trace(go.Scatter(
    x=df_pareto['product_id'],
    y=df_pareto['ca_cumulee'] * 100,
    name='Part cumulée (%)',
    yaxis='y2',
    mode='lines+markers'
))

# Axes + ordre de l'axe X selon ca_cumulee croissant
fig.update_layout(
    title="Diagramme de Pareto du CA par produit",
    xaxis=dict(title="Produit",
               categoryorder='array',
               categoryarray=df_pareto['product_id'].tolist()),
    yaxis=dict(title="Chiffre d'affaires"),
    yaxis2=dict(title='Part cumulée (%)', overlaying='y', side='right', range=[0, 100])
)

fig.show()


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.2 - Analyse des ventes en quantité</h3>
</div>

In [ ]:
#####################################
# Palmarès des articles en quantité #
#####################################

#Effectuer le tri dans l'ordre décroissant de quantités vendues du dataset df_merge
df_statsv=df_stats.sort_values(by='total_sales', ascending=False).copy()
#Réinitialiser l'index du dataset par un reset_index
df_statsv.reset_index(drop=True, inplace=True)
#Afficher les 20 premiers articles en quantité
df_statsv.head(20)

In [ ]:
#Graphique en barre des 20 premiers articles avec plotly express
df_statsv['xlabel'] = df_statsv['product_id'].astype(str).iloc[0:20]
fig = px.bar(df_statsv.iloc[0:20], x='xlabel', y='total_sales',
             color='product_type', 
             color_discrete_map=couleurs_produits,
             text='price',
             title='Classement des 20 articles les plus vendus ',
             labels={'xlabel':'Article', 'total_sales':"Quantité vendue"})
fig.update_xaxes(categoryorder='array', categoryarray=df_statsv['xlabel'])
fig.update_layout(legend_title_text='Type de produit')
fig.update_traces(texttemplate="%{text:.2f}€")
fig.show()

In [ ]:
df_statsv.head(20).describe()

In [ ]:
#############################
# Calculer le 20 / 80 en CA #
#############################

#Créer une colonne calculant la part en quantité de la ligne dans le dataset
df_statsv['part_qte']=df_statsv['total_sales']/df_statsv['total_sales'].sum()
#Créer une colonne réalisant la somme cumulative de la colonne précedemment créée
df_statsv['part_cumulee']=df_statsv['part_qte'].cumsum()
df_statsv.head(2)

In [ ]:
#Grâce aux deux colonnes créées précedemment, calculer le nombre d'articles représentant 80% des ventes en quantité
vente80=df_statsv[df_statsv['part_cumulee']<=0.8].shape[0]
print("Nombre d'articles représentant 80% des ventes en quantité: {}".format(vente80))
#Afficher la proportion que représente ce groupe d'articles dans le catalogue entier du site web
print("Proportion des articles représentant 80% des ventes en quantité dans le catalogue: {:.2f} %".format((vente80/df_statsv.shape[0])*100))

In [ ]:
# Sous-ensemble (<= 80%)
df80 = df_statsv.loc[df_statsv['part_cumulee'] <= 0.8]

# Proportion par product_type (dans ce sous-ensemble)
prop = df80['product_type'].value_counts(normalize=True)
prop_pct = (prop * 100).round(1)

print(prop_pct)

#Nuage de point
fig = px.scatter(df80, 
                 x='price', 
                 y='total_sales',
                 color='product_type',
                 color_discrete_map=couleurs_produits,
                 title='Ventes globales des produits selon leu prix',
                 labels={'total_sales':'Quantité vendue', 'price':"Prix en euros"},
                 hover_data=['product_id']
                )
fig.show()

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.3 - Analyse des stocks</h3>
</div>

In [ ]:
######################################
# Calculer le nombre de mois de stock #
######################################

for col in ['stock_quantity','total_sales','purchase_price']:
    df_stats[col] = pd.to_numeric(df_stats[col], errors='coerce')

#Création de la colonne Rotation de stock
stock_initial =(df_stats['stock_quantity']+df_stats['total_sales'])*df_stats['purchase_price']  
stock_final=df_stats['stock_quantity']*df_stats['purchase_price']  
stock_moyen = (stock_initial + stock_final)/2

df_stats['rotation_stock']=(df_stats['total_sales']*df_stats['purchase_price'])/stock_moyen  
#Remplacement des "inf" par 0
df_stats['rotation_stock']=df_stats['rotation_stock'].replace([np.inf, -np.inf], 0)

#Effectuer le tri dans l'ordre décroissant du nombre de mois de stock dans le dataset df_merge 
df_stats['mois_stock']=12/df_stats['rotation_stock']
df_stats['mois_stock']=df_stats['mois_stock'].replace([np.inf, -np.inf],0)
df_mois_stock=df_stats.sort_values(by='mois_stock', ascending=False).reset_index(drop=True).copy()
df_mois_stock.head(30)

In [ ]:
#Graphique en barre du flop 20 des produits qui ont le plus de mois de stock
df_mois_stock['xlabel']=df_mois_stock['product_id'].astype(str)
fig = px.bar(df_mois_stock.iloc[:20], x='xlabel', y='mois_stock',
            text='price',
            color='product_type', 
            color_discrete_map=couleurs_produits,
             title='Classement des 20 articles ayant le plus de mois en stock',
             labels={'xlabel':'Article', 'mois_stock':"mois de stock"})

fig.update_traces(texttemplate="%{text:.2f}€")
fig.update_xaxes(categoryorder='array', categoryarray=df_mois_stock['xlabel'])
fig.update_layout(legend_title_text='Type de produit')
fig.show()

In [ ]:
####################################
# Valorisation des stocks en euros #
####################################

#Création de la colonne Valorisation des stocks en euros
df_stats['Valorisation_stock_euros']=df_stats['stock_quantity']*df_stats['purchase_price']
#Calculer la somme de la colonne "Valorisation_stock_euros"
val_stock=df_stats['Valorisation_stock_euros'].sum()
print("La valorisation des stocks en euros est de: {:.2f} ".format(val_stock))

In [ ]:
#Graphique de la valorisation de stock des produits ayant le plus de mois en stock
df_valo_stock=df_stats.sort_values(by='mois_stock', ascending=False).iloc[:20].reset_index(drop=True).copy()
df_valo_stock=df_valo_stock.sort_values(by='Valorisation_stock_euros', ascending=False).reset_index(drop=True).copy()
df_valo_stock['xlabel']=df_valo_stock['id_web'].astype(str)
fig = px.bar(df_valo_stock.iloc[:20], x='xlabel', y='Valorisation_stock_euros',
             text='price',
            color='product_type', 
            color_discrete_map=couleurs_produits,
             title='Valorisation du stock des 20 articles ayant le plus de mois en stock',
             labels={'xlabel':'Article', 'Valorisation_stock_euros':"Valorisation de stock"})
             
fig.update_xaxes(categoryorder='array', categoryarray=df_valo_stock['xlabel'])
fig.update_layout(legend_title_text='Type de produit')
fig.update_traces(texttemplate="%{text:.2f}€")
fig.show()

Valeur_stock20=df_valo_stock['Valorisation_stock_euros'].sum()
print("La valorisation des stocks des 20 articles ayant le plus de mois en stock est de: {:.2f} ".format(Valeur_stock20))
print("Cela représente {:.2f} % de la valorisation totale des stocks".format((Valeur_stock20/val_stock)*100))

In [ ]:
#Pie chart de la valorisation des stocks en fonction de la valeur globale et des 20 articles ayant le plus de mois en stock
labels = ['20 articles avec le plus de mois en stock', 'Reste du stock']
values = [Valeur_stock20, val_stock - Valeur_stock20]
fig = px.pie(
    names=labels,
    values=values,
    color=labels,
    color_discrete_map={
        '20 articles avec le plus de mois en stock': 'saddlebrown',
        'Reste du stock': 'lightgrey'},
    title='Répartition de la valorisation des stocks'
)
fig.update_traces(textposition='inside', textinfo='percent')
fig.show()



In [ ]:
##############################################
# Valorisation du nombre de produits en stock #
##############################################

#Calculer la somme de la colonne stock quantity
total_stock=df_stats['stock_quantity'].sum()
print("Le nombre total de produits en stock est de: {} ".format(total_stock))

In [ ]:
#Graphique de la valorisation du nombre de produit en stock des produits ayant le plus de mois en stock

df_total_stock=df_valo_stock.sort_values(by='stock_quantity', ascending=False).reset_index(drop=True).copy()
df_total_stock['xlabel']=df_valo_stock['id_web'].astype(str)
fig = px.bar(df_total_stock.iloc[:20], x='xlabel', y='stock_quantity',
              text='price',
            color='product_type', 
            color_discrete_map=couleurs_produits,
             title='Nombre de produits en stock des 20 articles ayant le plus de mois en stock',
             labels={'xlabel':'Article', 'stock_quantity':"Nombre de produits en stock"})
fig.update_xaxes(categoryorder='array', categoryarray=df_total_stock['xlabel'])
fig.update_layout(legend_title_text='Type de produit')
fig.update_traces(texttemplate="%{text:.2f}€")
fig.show()

Total_stock20=df_total_stock['stock_quantity'].sum()
print("Le nombre de produits en stock des 20 articles ayant le plus de mois en stock est de: {:.2f} ".format(Total_stock20)) 
print("Cela représente {:.2f} % du nombre total de produits en stock".format((Total_stock20/total_stock)*100))

In [ ]:
dfp = df_stats.copy()

#prix unitaire vs valorisation (taille = quantité, couleur = catégorie)
fig = px.scatter(
    dfp,
    x='purchase_price',
    y='Valorisation_stock_euros',
    size='stock_quantity',
    color='product_type',
    color_discrete_map=couleurs_produits,
    title="Valorisation du stock en fonction du prix d'achat des produits",
    labels={'purchase_price':'Prix d’achat','Valorisation_stock_euros':'Valorisation de stock (€)'}
)
fig.update_layout(legend_title_text='Type de produit')
fig.show()

#Valorisation du stock par type de produit
df_cat = (dfp.groupby('product_type', as_index=False)['Valorisation_stock_euros']
            .sum()
            .sort_values('Valorisation_stock_euros', ascending=False))

fig = px.bar(
    df_cat,
    x='product_type',
    y='Valorisation_stock_euros',
    color='product_type',
    color_discrete_map=couleurs_produits,
    title='Valorisation de stock par type de produit',
    labels={'product_type':'Type de produit','Valorisation_stock_euros':'Valorisation de stock (€)'}
)
fig.update_layout(legend_title_text='Type de produit', showlegend=False)
fig.show()



<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.4 - Analyse du taux de marge</h3>
</div>

In [ ]:
############################
# Analyse du taux de marge #
############################

#Création de la colonne Prix HT
df_stats['prix_ht']=round((df_stats['price']/1.2),2)
#Création de la colonne Taux de marge
df_stats['taux_marge']=round((df_stats['price']*100/df_stats['purchase_price'])-100,2)
#Afficher le prix minimum de la colonne "taux_marge"
print("Taux de marge min: {} %".format(round(df_stats['taux_marge'].min(),2)))
#Afficher le prix maximum de la colonne "taux_marge"
print("Taux de marge max: {} %".format(round(df_stats['taux_marge'].max(),2)))

df_stats.head(5)

In [ ]:
#Affichage de la ligne avec un taux de marge inférieur à 0
taux_marge_neg=df_stats[df_stats['taux_marge']<0]
taux_marge_neg  

In [ ]:
#Création d'un dataframe avec les taux positifs
df_taux_positif=df_stats[df_stats['taux_marge']>=0]
#Afficher le prix minimum de la colonne "taux_marge"
print("Taux de marge minimum: {}".format(df_taux_positif['taux_marge'].min())) 

In [ ]:
#Création d'un dataframe avec le taux de marge moyen par type de produit
df_produit=round(df_taux_positif.groupby('product_type')['taux_marge'].mean().reset_index().sort_values(by='taux_marge', ascending=False), 2)
df_produit.rename(columns={'taux_marge':'taux_marge_moyen'}, inplace=True)
df_produit['Ecart_type']=round(df_taux_positif.groupby('product_type')['taux_marge'].std().reset_index().sort_values(by='taux_marge', ascending=False)['taux_marge'],2)   
df_produit.head(6)

In [ ]:
#Affichage dans un graphique du taux de marge par type de produit
from plotly import graph_objects as go

fig = go.Figure(go.Funnel(
    y = df_produit['product_type'],
    x = df_produit['taux_marge'],
    textposition='inside',
    texttemplate="%{value:.2f}%",
    opacity = 0.65, marker = {"color": ["saddlebrown", "darkgoldenrod", "palegoldenrod", "darkred","lightgoldenrodyellow", "olive"]}))
fig.update_layout(title="Taux de marge appliqué selon les produits")
fig.show()

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.5 - Analyse des corrélations entre les variables stock, sales et price</h3>
</div>

In [ ]:
############################
# Analyse des corrélations #
############################

#Importation de Seaborn

#Création d'une heatmap de corrélation avec les variables stock, sales et price
corr = df_stats[['stock_quantity','total_sales','price']].corr()

ax = sb.heatmap(data=corr, annot=True, fmt='.2g', linewidths=0.5, linecolor='white', cbar=True)
plt.title('Corrélations : stock / ventes / prix')
plt.show()


In [ ]:
#On peut également créer un mask pour n'afficher qu'une demi heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
ax = sb.heatmap(data=corr, annot=True, fmt='.2g', linewidths=0.5, linecolor='white', cbar=True, mask=mask)
plt.title('Corrélations : stock / ventes / prix')
plt.show()

*Que peut-on conclure des corrélations ?*

Trop peu de corrélation entre les variables, elles ne s'influencent donc pas entre elles.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.6 - Mise à disposition de la nouvelle table sur un fichier Excel</h3>
</div>

In [ ]:
#Mettre le dataset df_merge sur un fichier Excel
#Cette étape peut être utile pour partager le résultat du dataset obtenu avec les équipes.  
df_stats.to_excel("C:/Users/chris/Documents/GitHub/OC/P6/data_raw/df_stats.xlsx", index=False)